# Covered calls after completed revolving-credit renewals

**Predicted contrast:** covered-call incremental net value over stock, post-disclosure minus five-session pre-disclosure baseline, is negative at 21 sessions.

This is an explicitly authorized retrospective **before/after baseline**, not clean ordinary-day alpha. The baseline holds through the disclosure and is not a prospective entry signal. Discovery: March 2022–December 2023; historical validation: 2024–2025. These expanded issuers exclude the starter 100 issuers. No judges' sealed window is used.

The original notebook and harness remain unchanged. All rules were frozen before empirical P&L. See the frozen hypothesis, effect registration, saved inputs, exclusion counts and script alongside this notebook.


In [1]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
ROOT = Path.cwd()
assert (ROOT / 'credit_renewal_test.py').exists(), 'Run this notebook from the repository root.'
OUT = ROOT / 'credit_prebaseline_v3_candidate' / 'experiment'
def run_script(*args):
    result = subprocess.run([sys.executable, *args], cwd=ROOT, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError('Command failed. Inspect local error without publishing credentials: ' + str(args))
    print(result.stdout)
run_script('test_credit_renewal_test.py')
run_script('credit_renewal_test.py', 'register')
import hashlib
frozen = json.loads((OUT / 'analysis_implementation_freeze.json').read_text())
assert all(hashlib.sha256((ROOT / name).read_bytes()).hexdigest() == digest for name, digest in frozen['files'].items())


Passed assignment/dividend/expiry accounting, IV inversion and finite-t critical-value checks; no empirical outcomes read.




## Count and collect before outcomes

Collection resumes from saved per-event checkpoints and reuses exact option entries. Missing exits, split changes, dividend failures and supplemental-baseline failures remain exclusions. Existing primary contracts are fixed across horizons. One covered contract represents 100 funded shares; displayed quote sizes do not establish scalable capacity.


In [2]:
run_script('credit_renewal_test.py', 'collect')
counts = pd.read_csv(OUT / 'coverage_counts.csv')
print(counts.to_string(index=False))
primary_counts = counts[(counts.baseline_sessions == 5) & (counts.horizon.astype(str) == '21')]
assert len(primary_counts) == 2
print('Primary return-test gate:', bool(primary_counts.gate.all()))
run_script('audit_credit_renewal_inputs.py')


1655 sessions in the calendar; last completed session 2026-10-02. Sessions per year: 2022: 251, 2023: 250, 2024: 252, 2025: 250, 2026: 251
All frozen input retrieval complete; counts saved before outcomes

    window  baseline_sessions horizon  examined  usable_pairs  dependence_groups  gate  returns_calculated
 discovery                  5       1        40            40                 18  True               False
 discovery                  5       2        40            40                 12  True               False
 discovery                  5       3        40            39                 12 False               False
 discovery                  5       5        40            39                 10 False               False
 discovery                  5      10        40            39                 10 False               False
 discovery                  5      21        40            40                  7  True               False
 discovery                  5      42        

## Frozen effect test

Sell at bid, close at ask, $0.65 per contract per side. Report incremental P&L divided by entry stock reference, not premium ROI. Conservative plausible early-assignment bounds deliver stock at strike and account for foregone later dividends; an assigned stock leg no longer cancels. Expiry uses intrinsic settlement at expiry-session close. Quote fills, stock reference prices and modeled assignments remain assumptions.

Intervals use 100,000 dependence-component bootstraps and finite-component t intervals, both Bonferroni-adjusted across the retained 120-comparison family. Minimum 40 pairs/five components; other horizons and small sensitivities are exploratory or descriptive. Wide or unavailable intervals imply inconclusive evidence, not proof of no effect.


In [3]:
run_script('credit_renewal_test.py', 'analyze')
results = json.loads((OUT / 'results.json').read_text())
rows = []
for r in results:
    rows.append(dict(window=r['window'], horizon=r['horizon'], n=r['n'],
        groups=r.get('dependence_groups'), gate=r.get('gate'),
        mean_difference_pp=None if r.get('mean') is None else 100*r['mean'],
        bootstrap_interval_pp=None if not r.get('bootstrap_interval') else [100*x for x in r['bootstrap_interval']],
        finite_t_interval_pp=None if not r.get('t_interval') else [100*x for x in r['t_interval']]))
print(pd.DataFrame(rows).to_string(index=False))


1655 sessions in the calendar; last completed session 2026-10-02. Sessions per year: 2022: 251, 2023: 250, 2024: 252, 2025: 250, 2026: 251
discovery 1 n 40 mean -0.003163833764131115 groups 18
discovery 2 n 40 mean 0.0006279224941569125 groups 12
discovery 3 n 39 mean -0.0001358824563147045 groups 12
discovery 5 n 39 mean -0.0021887519913226486 groups 10
discovery 10 n 39 mean 0.000763446401650001 groups 10
discovery 21 n 40 mean -0.001037481128635982 groups 7
discovery 42 n 33 mean -0.012558351098271026 groups 2
discovery 63 n 27 mean -0.02570857154580362 groups 1
discovery exp n 40 mean -0.004016547992427111 groups 1
validation 1 n 39 mean -0.004191786647377966 groups 21
validation 2 n 37 mean -0.006216557891358824 groups 16
validation 3 n 39 mean -0.00723090567146951 groups 14
validation 5 n 39 mean -0.01297802934476726 groups 12
validation 10 n 37 mean -0.005363138542014832 groups 10
validation 21 n 40 mean -0.00512810631272406 groups 8
validation 42 n 29 mean 0.006174812442460345 

In [4]:
for r in results:
    if r['horizon'] != '21': continue
    print(); print('PRIMARY', r['window'])
    print(json.dumps(r, indent=2))
print(); print('Conclusion:', json.loads((OUT / 'conclusion.json').read_text())['conclusion'])



PRIMARY discovery
{
  "window": "discovery",
  "horizon": "21",
  "n": 40,
  "dependence_groups": 7,
  "mean": -0.001037481128635982,
  "bootstrap_interval": [
    -0.013519389239865964,
    0.011052135607557446
  ],
  "t_interval": [
    -0.03114731375846856,
    0.029072351501196594
  ],
  "gate": true,
  "assignment_lower": {
    "n": 40,
    "dependence_groups": 7,
    "mean": -0.05697610834481346,
    "bootstrap_interval": [
      -0.1173576151647475,
      -0.0074595464666148996
    ],
    "t_interval": [
      -0.1709137528840242,
      0.05696153619439726
    ],
    "gate": true
  },
  "assignment_upper": {
    "n": 40,
    "dependence_groups": 7,
    "mean": 0.06432440175958416,
    "bootstrap_interval": [
      0.02683357378848041,
      0.09860459972059307
    ],
    "t_interval": [
      -0.003311006150497481,
      0.1319598096696658
    ],
    "gate": true
  },
  "event": {
    "value": -0.029777210705939895,
    "entry_credit": 0.05371334843312738,
    "entry_mid": 0.06

## Mechanism and sensitivities

Report entry credits, subsequent absolute movements and upside/downside tails, and option-exit intrinsic/time value. American call IV uses a disclosed continuous trailing-dividend approximation, rather than vendor IV; failed inversions stay excluded. Regression adjusts maturity, recent realized volatility and actual strike positioning. IV dependence includes both holding and RV estimation intervals, potentially leaving too few independent groups for inference.

Sensitivities were specified before results: three/seven-session baselines, same-strike subset, commissions, adverse spread slippage, assignment fees, cash interest, bid/ask IV, zero/6% rates, zero dividends and 400-step trees. The matched cohort is never expanded after seeing P&L. A narrower P&L contrast alone cannot establish the financing-resolution mechanism.


In [5]:
sensitivity = json.loads((OUT / 'sensitivity_results.json').read_text())
print(pd.DataFrame([dict(window=r['window'], variant=r['variant'], n=r['n'],
    groups=r.get('dependence_groups'), gate=r.get('gate'),
    mean_difference_pp=None if r.get('mean') is None else 100*r['mean']) for r in sensitivity]).to_string(index=False))
mechanism = json.loads((OUT / 'mechanism_results.json').read_text())
print(pd.DataFrame([{k:v for k,v in r.items() if k != 'inversion_exclusions'} for r in mechanism]).to_string(index=False))
for window in ['discovery','validation']:
    print(window, json.loads((OUT / f'{window}_influence.json').read_text()))


    window       variant  n  groups  gate  mean_difference_pp
 discovery     baseline3 38       7 False            0.627363
 discovery     baseline7 22       7 False            0.499273
 discovery   commission1 40       7  True           -0.103494
 discovery slippage10pct 40       7  True           -0.129674
 discovery slippage25pct 40       7  True           -0.168563
 discovery   assignment5 40       7  True           -0.103748
 discovery  assignment15 40       7  True           -0.103748
 discovery cash_interest 40       7  True           -0.103748
 discovery   same_strike 24       8 False           -0.722889
validation     baseline3 34       8 False            0.079544
validation     baseline7 32       8 False           -0.163084
validation   commission1 40       8  True           -0.513396
validation slippage10pct 40       8  True           -0.575425
validation slippage25pct 40       8  True           -0.669346
validation   assignment5 40       8  True           -0.512811
validati

## Interpretation

Support requires negative corrected discovery and historical-validation intervals, robust assignment bounds and corresponding IV mechanism evidence. Positive significant replicated contrast contradicts the predicted direction; inconsistent, imprecise or mechanism-incomplete results are inconclusive. Historical validation here is not a pristine sealed-window replication. Seven/eight primary dependence groups do not guarantee adequate power under this multiplicity correction.

This notebook provides a before/after research test, not a portfolio trading recommendation. The pre-event entry requires an independent advance disclosure calendar to be tradable. Read the generated report and preserve failures rather than choosing favorable horizons or parameters.
